# Notebook 09 — Brand Benchmarking Scorecard

**Project:** Starbucks Customer Voice Intelligence: A U.S. Coffee Chain Market Study  
**Purpose:** Score Starbucks against Dunkin' and the independent café market across five operational dimensions, identifying where the performance gap is largest and where it is narrowest.

## 0. Environment setup

In [1]:
import sys
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Environment ready.")

Environment ready.


## 1. Load data

In [2]:
df = pd.read_parquet(INTERIM_DIR / "analysis_ready.parquet")
print(f"Total reviews: {len(df):,}")
print(df["brand_category"].value_counts().to_string())

Total reviews: 381,999
brand_category
Independent Café    362334
Starbucks            11675
Dunkin'               6866
Dutch Bros             925
The Coffee Bean        128
Peet's Coffee           71


## 2. Overall performance table

Three brands are included in the scorecard: **Starbucks** (primary subject), **Dunkin'** (primary chain competitor, 6,866 reviews), and **Independent Café** (market benchmark, 362,334 reviews). Dutch Bros, The Coffee Bean, and Peet's Coffee are excluded due to insufficient sample size.

In [3]:
scored_brands = ["Starbucks", "Dunkin'", "Independent Café"]

rows = []
for b in scored_brands:
    g = df[df["brand_category"] == b]
    rows.append({
        "Brand":         b,
        "Reviews":       len(g),
        "Avg Stars":     round(g["review_stars"].mean(), 2),
        "% Positive":    round((g["star_tier"] == "Positive").mean() * 100, 1),
        "% Critical":    round((g["star_tier"] == "Critical").mean() * 100, 1),
        "Avg Sentiment": round(g["sentiment_score"].mean(), 3),
    })

tbl = pd.DataFrame(rows)
print(tbl.to_string(index=False))

           Brand  Reviews  Avg Stars  % Positive  % Critical  Avg Sentiment
       Starbucks    11675       2.90        42.4        47.6          0.277
         Dunkin'     6866       2.05        21.5        71.2         -0.020
Independent Café   362334       4.01        73.7        17.2          0.706


## 3. Scorecard — topic positive rate rose chart

The rose chart scores each brand across five operational topic dimensions. For each dimension, the metric is the **positive review rate** within that topic — the share of reviews mentioning that topic that are rated 4 or 5 stars. A higher score indicates stronger performance in that area.

Dimensions: Service · Food Quality · Ambiance · Wait Time · Cleanliness

In [4]:
topic_dims = ["Service", "Food Quality", "Ambiance", "Wait Time", "Cleanliness"]

scores = {}
for b in scored_brands:
    g = df[df["brand_category"] == b]
    rates = {}
    for t in topic_dims:
        # Multi-label: count reviews that mention this topic (may also carry other tags)
        mask  = g["topic_tags"].str.contains(t, regex=False)
        total = mask.sum()
        pos   = (mask & (g["star_tier"] == "Positive")).sum()
        rates[t] = round(pos / total * 100, 1) if total > 0 else 0
    scores[b] = rates

score_df = pd.DataFrame(scores).T
score_df["Composite"] = score_df.mean(axis=1).round(1)
print("Topic Positive Rate by Brand (scored 0–100):")
print(score_df.to_string())

Topic Positive Rate by Brand (scored 0–100):
                  Service  Food Quality  Ambiance  Wait Time  Cleanliness  Composite
Starbucks            44.4          39.5      62.2       39.9         47.2       46.6
Dunkin'              22.1          22.2      42.2       25.2         27.2       27.8
Independent Café     71.5          73.9      78.8       65.4         62.4       70.4


In [5]:
# close the polygon for each trace
theta = topic_dims + [topic_dims[0]]

brand_styles = {
    "Starbucks":        {"color": "#00704A", "dash": "solid"},
    "Dunkin'":          {"color": "#d62728", "dash": "dash"},
    "Independent Café": {"color": "#aaaaaa", "dash": "dot"},
}

fig = go.Figure()
for brand, style in brand_styles.items():
    vals = [scores[brand][d] for d in topic_dims] + [scores[brand][topic_dims[0]]]
    fig.add_trace(go.Scatterpolar(
        r=vals,
        theta=theta,
        name=brand,
        mode="lines+markers",
        line=dict(color=style["color"], dash=style["dash"], width=2.5),
        marker=dict(color=style["color"], size=8),
        fill="toself",
        fillcolor=style["color"],
        opacity=0.12,
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 100],
            tickvals=[0, 25, 50, 75, 100],
            ticksuffix="%",
            gridcolor="#dddddd",
            linecolor="#dddddd",
        ),
        angularaxis=dict(
            gridcolor="#dddddd",
            linecolor="#dddddd",
        ),
    ),
    title=dict(
        text="Brand Scorecard — Topic Positive Rate (%) by Brand",
        font=dict(size=16),
        x=0.5,
    ),
    legend=dict(orientation="h", yanchor="bottom", y=-0.18, xanchor="center", x=0.5),
    paper_bgcolor="white",
    width=650, height=600,
    margin=dict(t=80, b=90, l=60, r=60),
)
fig.write_html(str(FIGURES_DIR / "09_scorecard_rose.html"))
fig.show()

## Key findings

**Both coffee chains trail the independent café market on every operational dimension.**
Independent cafés achieve positive rates of 62–79% across all five topic categories. Starbucks and Dunkin' both fall well below this benchmark, confirming that the performance gap is a chain-format problem, not unique to either brand.

**Starbucks outperforms Dunkin' across all five dimensions.**
Despite Starbucks's below-market overall rating of 2.90 stars, it consistently scores higher than Dunkin' in every topic category. Dunkin' averages 2.05 stars overall and posts a 71.2% critical review rate — the weakest performance of any brand with sufficient sample volume in this dataset.

**Ambiance is the strongest dimension for all three brands.**
Starbucks achieves 62.2% positive rate on Ambiance — its highest score and the closest it comes to the market benchmark (78.8%). In-store environment is where chain brands compete most effectively with independent cafés.

**Cleanliness has improved under multi-label tagging but remains weak.**
Starbucks scores 47.2% on Cleanliness positive rate (1,609 reviews); Dunkin' scores 27.2%. Independent cafés score 62.4%. Under single-label tagging, Cleanliness volumes were too small to draw conclusions; with multi-label matching, the sample is large enough to confirm both chains underperform independents on hygiene.

**Composite scores (average topic positive rate):**
- Independent Café: 70.4%
- Starbucks: 46.6%
- Dunkin': 27.8%

---

**Next: Notebook 10 — Time Pattern Analysis**